# Indexing: Text Embedding with Ollama

In [31]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters.markdown import MarkdownHeaderTextSplitter
from langchain_text_splitters.character import CharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings
import numpy as np

In [32]:
loader_docx = Docx2txtLoader("./Introduction-to-Data-and-Data-Science-2.docx")
pages = loader_docx.load()

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "Course Title"),
        ("##", "Lecture Title")
    ]
)

pages_md_split = md_splitter.split_text(pages[0].page_content)

for i in range(len(pages_md_split)):
    pages_md_split[i].page_content = ' '.join(pages_md_split[i].page_content.split())


char_splitter = CharacterTextSplitter(
    separator='.',
    chunk_size=500,
    chunk_overlap=50
)

pages_char_split = char_splitter.split_documents(pages_md_split)

In [33]:
pages_char_split

[Document(metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Analysis vs Analytics'}, page_content='Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both. So, let’s clear this up, shall we? First, we will start with analysis'),
 Document(metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Analysis vs Analytics'}, page_content='Consider the following… You have a huge dataset containing data of various types. Instead of tackling the entire dataset and running the risk of becoming overwhelmed, you separate it into easier to digest chunks and study them individu

In [34]:
embedding = OllamaEmbeddings(model = "nomic-embed-text")

In [35]:
pages_char_split[18]

Document(metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need'}, page_content='More importantly, it will be sufficient for your need to create quick and accurate analyses. However, if your theoretical preparation is strong enough, you will find yourself restricted by software. Knowing a programming language such as R and Python, gives you the freedom to create specific, ad-hoc tools for each project you are working on')

In [36]:
vector_1 = embedding.embed_query(pages_char_split[3].page_content)
vector_2 = embedding.embed_query(pages_char_split[5].page_content)
vector_3 = embedding.embed_query(pages_char_split[18].page_content)

In [37]:
len(vector_1), len(vector_2), len(vector_3)

(768, 768, 768)

In [42]:
from numpy.linalg import norm
import numpy as np

def normalize(v):
    return v / np.linalg.norm(v)

v1 = normalize(vector_1)
v2 = normalize(vector_2)
v3 = normalize(vector_3)

np.dot(v1, v2), np.dot(v1, v3), np.dot(v2, v3)

(np.float64(0.7847997745599395),
 np.float64(0.7218966708110345),
 np.float64(0.5757039790628574))